# CourtLens GPU eval (generated by scripts/kaggle/build_kaggle.py — do not edit here)
Runs the real backend pipeline (`app.scripts.run_offline`) on facility CCTV, DFL clips, and a synthetic 3-camera check. Accelerator: **GPU T4**.

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile
from pathlib import Path

# System deps: ffmpeg (probe/sync), tesseract (jersey OCR — was missing in old runs)
subprocess.call("which ffmpeg ffprobe || (apt-get -qq update && apt-get -qq install -y ffmpeg)", shell=True)
subprocess.call("which tesseract || (apt-get -qq update && apt-get -qq install -y tesseract-ocr)", shell=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *['ultralytics==8.4.183', 'lap>=0.5.12', 'pydantic-settings==2.6.1', 'pytesseract>=0.3.10', 'torchreid==0.2.5', 'umap-learn>=0.5.6', 'motmetrics>=1.4.0', 'remotezip>=0.12']])

# Kaggle auto-extracts uploaded zips; accept either the zip or the extracted tree
BACKEND = Path("/kaggle/working/backend")
if BACKEND.exists():
    shutil.rmtree(BACKEND)
code_zip = glob.glob("/kaggle/input/**/courtlens_backend.zip", recursive=True)
runner = glob.glob("/kaggle/input/**/app/scripts/run_offline.py", recursive=True)
if code_zip:
    zipfile.ZipFile(code_zip[0]).extractall(BACKEND)
elif runner:
    shutil.copytree(Path(runner[0]).parents[2], BACKEND)  # read-only input → writable copy
else:
    raise SystemExit("attach dataset muddasirr/courtlens-code")
print("code rev", (BACKEND / "GIT_REV").read_text())
mounted = (BACKEND / "BUILD_ID").read_text().strip() if (BACKEND / "BUILD_ID").exists() else "none"
assert mounted == "20261026-162309", f"stale code bundle {mounted} (expected 20261026-162309) — re-push after the dataset finishes processing"

In [ ]:
import torch
assert torch.cuda.is_available(), "Settings → Accelerator → GPU T4, then re-run"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

N_DFL = int(os.environ.get("COURTLENS_N_DFL", "0"))  # DFL 30 s clips already measured (v9)
OUT = Path("/kaggle/working/eval"); OUT.mkdir(exist_ok=True)
mp4 = {}
for p in Path("/kaggle/input").rglob("*.mp4"):
    mp4.setdefault(p.name, p)
weights = next(iter(Path("/kaggle/input").rglob("yolov5x_football.pt")), None)

# Corpus config: one general detector for every video (futsal CCTV, futsal broadcast, DFL)
PRIMARY = os.environ.get("COURTLENS_DETECTOR", "yolov8m.pt")  # best on all 5 futsal videos (v6)
env = dict(os.environ, PYTHONPATH=str(BACKEND), VISION_DEVICE="0", VISION_TARGET_FPS="10",
           VISION_IMGSZ=os.environ.get("COURTLENS_IMGSZ", "0"), VISION_JERSEY_OCR="1",
           VISION_REID="1", VISION_TEAM_EMBED="1", YOLO_MODEL=PRIMARY)

def run(slug, cams, **extra):
    out = OUT / slug
    cmd = [sys.executable, "-m", "app.scripts.run_offline", "--out", str(out), "--title", slug]
    for label, path in cams.items():
        cmd += ["--camera", f"{label}={path}"]
    t0 = time.time()
    r = subprocess.run(cmd, cwd=BACKEND, env=dict(env, **extra), capture_output=True, text=True)
    (OUT / f"{slug}.log").write_text(r.stdout + "\n" + r.stderr)
    row = {"slug": slug, "cameras": sorted(cams), "rc": r.returncode, "seconds": round(time.time() - t0)}
    if r.returncode == 0:
        s = json.loads((out / "summary.json").read_text())
        row.update(counts=s["counts"], sync=s["sync"], vision=s["stages"].get("detect_track", {}).get("vision_stats"))
        for big in ("courtlens.db", "storage"):  # keep the output small
            p = out / big
            shutil.rmtree(p) if p.is_dir() else p.unlink(missing_ok=True)
    else:
        row["error"] = r.stderr[-2000:]
    print(json.dumps({k: row.get(k) for k in ("slug", "rc", "seconds", "counts", "sync")}))
    return row

index = []
# Detector comparison on futsal (the football fine-tune finds ~0.2 players/frame on CCTV)
FUTSAL = [n for n in os.environ.get("COURTLENS_FUTSAL", "").split(",") if n]
DETECTORS = [d for d in os.environ.get("COURTLENS_DETECTORS", PRIMARY).split(",") if d]
for name in FUTSAL:
    if name not in mp4:
        continue
    for det in DETECTORS:
        local = next(iter(Path("/kaggle/input").rglob(det)), None)
        index.append(run(f"{name.removesuffix('.mp4')}__{det.removesuffix('.pt')}", {"A": mp4[name]},
                         YOLO_MODEL=str(local or det)))
        (OUT / "eval_index.json").write_text(json.dumps(index, indent=2))

# DFL 30 s clips (tracking / kit-split stress; dataset has no event labels)
dfl = sorted(n for n in mp4 if n[:1] in "ABCDE" and "(" in n)  # e.g. "A1606b0e6_0 (11).mp4"
step = max(1, len(dfl) // max(1, N_DFL)) if N_DFL else 0
for name in (dfl[::step][:N_DFL] if N_DFL else []):
    slug = "dfl_" + name.replace(" ", "").replace("(", "_").replace(")", "").removesuffix(".mp4")
    index.append(run(slug, {"A": mp4[name]}, MIN_VIDEO_DURATION_S="20"))
(OUT / "eval_index.json").write_text(json.dumps(index, indent=2))

In [ ]:
# Product candidate on the labelled futsal clip: trained ball detector + futsal-fine-tuned T-DEED
subprocess.call([sys.executable, "-m", "pip", "install", "-q", "SoccerNet==0.1.62", "tabulate"])
if not Path("/kaggle/working/tdeed").exists():
    subprocess.check_call("git clone -q https://github.com/arturxe2/T-DEED.git /kaggle/working/tdeed && "
                          "git -C /kaggle/working/tdeed checkout -q d953b4de8a2ad941c565e1683c5b07255cbe644e", shell=True)
w = next(Path("/kaggle/input").rglob("tdeed_futsal.pt"))
ball = next(Path("/kaggle/input").rglob("ball_yolov8m.pt"))
# A/B the ball detector under the futsal spotter (trained ball found 17 % vs COCO 48 % on this clip)
for tag, extra in [("futsal_coco", {}), ("futsal_ball", {"VISION_BALL_MODEL": str(ball)})][: int(os.environ.get("COURTLENS_FUTSAL_VARIANTS", "1"))]:
    index.append(run(f"test_video_2__{tag}", {"A": mp4["test_video_2.mp4"]}, TDEED_REPO="/kaggle/working/tdeed",
                     TDEED_WEIGHTS=str(w), FFMPEG="ffmpeg",
                     TDEED_THRESHOLDS='{"pass": 0.4, "shot": 0.15, "dribble": 0.3}', **extra))
(OUT / "eval_index.json").write_text(json.dumps(index, indent=2))

In [ ]:
# 5) One-file bundles — `kaggle kernels output` fetches files one by one (thousands of thumbs)
import zipfile
with zipfile.ZipFile("/kaggle/working/results.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob("*"):
        if p.is_file() and (p.suffix in {".json", ".log"} or p.name == "jersey_resnet18.pt"):
            zf.write(p, p.relative_to(OUT.parent))
with zipfile.ZipFile("/kaggle/working/thumbs.zip", "w", zipfile.ZIP_STORED) as zf:
    for p in OUT.rglob("thumbs/*/*.jpg"):
        zf.write(p, p.relative_to(OUT.parent))
print("bundles", Path("/kaggle/working/results.zip").stat().st_size // 1_000_000, "MB")